# Optimizing parameters for the Zener function used to describe rock stress

In [ ]:
!pip install matplotlib numpy pandas pygad tqdm

In [ ]:
from contextlib import contextmanager
import json
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pygad
from tqdm import tqdm

In [4]:
Np = 21  # Number of periods Tp
Nsamp = 1000  # Number of samples per period

Sig0 = 1
t0 = 0.0

def fSig(x, x1, Rs):
    return Rs * (x - x1)

def fEpsNL(Sig, EmNL, Alpha):
    return Sig / (EmNL * (1 + Alpha * Sig))

def fEps(x, x1, x2, Rs, Em, Tau_s, Tau_e):
    return (Rs / Em) * (
        (x2 - x1) + (Tau_s - Tau_e) * (np.exp(x2 / Tau_e) - np.exp(x1 / Tau_e)) * np.exp(-x / Tau_e)
    )

def fEps0(x, Rs, Em, Tau_s, Tau_e):
    return fEps(x, t0, x, Rs, Em, Tau_s, Tau_e)

def fEpsVE(x, i, Rsig, R, Tp, Em, Tau_s, Tau_e):
    epsVE = fEps0(x - (i - 1) * Tp, Rsig[i - 1] * R, Em, Tau_s, Tau_e)
    for k in range(1, i):
        epsVE += fEps(x, (k - 1) * Tp, k * Tp, Rsig[k - 1] * R, Em, Tau_s, Tau_e)
    return epsVE

In [5]:
def parse_signal_file(filename='Rsig-Samp.txt'):
    signals = {}
    header = None

    with open(filename) as f:
        f.readline()

        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split('\t')
            label, values = parts[0], parts[1:]
    
            try:
                int_values = [int(v) for v in values]
                if not int_values:
                    header = label
                    continue
                signals[f'{header} - {label}'] = int_values
            except ValueError:
                header = label

    return signals

In [6]:
@contextmanager
def change_dir(target_dir):
    prev_dir = os.getcwd()
    os.makedirs(target_dir, exist_ok=True)
    os.chdir(target_dir)
    try:
        yield
    finally:
        os.chdir(prev_dir)

In [7]:
class GASubject:
    
    def __init__(self, label, Rsig):
        self.label = label
        self.Rsig = Rsig
        self.load_testing_data()

    def load_testing_data(self):
        df_target = pd.read_csv(f'{self.label}/ZenerModel.dat', delim_whitespace=True)
        df_target['time'] = pd.to_numeric(df_target['time'], errors='coerce')
        self.df_target = df_target.dropna()
        self.df_target_sorted = self.df_target.sort_values('time')
    
        self.dt = self.df_target.iloc[1].time - self.df_target.iloc[0].time
        self.Tp = self.dt * Nsamp

        print(f'{self.label} loaded')

    def compute_all(self, params):
        Em, Tau_e, Tau_s, EmNL, Alpha = params
        
        t = 0.0
        FvalSig = 0.0

        Rsig = self.Rsig
        Tp = self.Tp
        R = Sig0 / Tp
        
        rows = []
    
        for i in range(1, Np + 1):
            for j in range(Nsamp):
                t += self.dt
                valSig = FvalSig + fSig(t, (i - 1) * Tp, Rsig[i - 1] * R)
                valEpsNL = fEpsNL(valSig, EmNL, Alpha)
                valEpsVE = fEpsVE(t, i, Rsig, R, Tp, Em, Tau_s, Tau_e)
                rows.append([
                    t,
                    valSig,
                    valEpsNL,
                    valEpsVE,
                    valEpsNL + valEpsVE,
                ])
            
            FvalSig = valSig
        
        return pd.DataFrame(rows, columns=['time', 'Stress', 'StrainNonLinEl', 'StrainViscEl', 'TotStrain'])
    
    def compute_and_merge(self, params):
        df = self.compute_all(params)
        df_sorted = df.sort_values('time')
        
        merged = pd.merge_asof(
            df_sorted, 
            self.df_target_sorted, 
            on='time', 
            direction='nearest', 
            tolerance=1e-8,
            suffixes=('_model', '_target'),
        )
        return merged

    def fitness_func(self, ga_instance, solution, solution_idx):
        merged = self.compute_and_merge(solution)
    
        match_ratio = len(merged) / len(self.df_target)
        if match_ratio < 0.95:
            return -1e9
    
        error = 0.0
        for col in ['TotStrain']:
            diff = merged[f'{col}_model'] - merged[f'{col}_target']
            error += np.sum(diff ** 2)
    
        return -error

    def on_generation(self):
        self.pbar.set_postfix(fitness=f'{self.ga.last_generation_fitness[0]:.4e}')
        self.pbar.n = self.ga.generations_completed
        self.pbar.refresh()

    def run_ga(self):
        num_generations = 30
        print(f'Starting GA for {self.label}...')
        
        # with tqdm(total=num_generations) as self.pbar:
        self.ga = pygad.GA(
            num_generations=num_generations,
            stop_criteria=['reach_-10', 'saturate_15'],
            num_parents_mating=5,
            fitness_func=self.fitness_func,
            sol_per_pop=20,
            num_genes=5,
            mutation_percent_genes=30,
            mutation_by_replacement=True,
            gene_space=[
                {'low': 0.1, 'high': 10.0},  # Em
                {'low': 10, 'high': 100},    # Tau_e
                {'low': 10, 'high': 100},    # Tau_s
                {'low': 0.01, 'high': 3},    # EmNL
                {'low': 0.1, 'high': 10},    # Alpha
            ],
            # on_generation=lambda _: self.on_generation(),
            save_best_solutions=True,
            parallel_processing=['process', 4],
        )
        
        self.ga.run()

    def save_result(self):
        with change_dir(self.label):
            solution, solution_fitness, _ = self.ga.best_solution()
            print(solution, solution_fitness)
            
            Em, Tau_e, Tau_s, EmNL, Alpha = solution
            with open('solution.json', 'w') as f:
                json.dump({
                    'Em': Em,
                    'Tau_e': Tau_e,
                    'Tau_s': Tau_s,
                    'EmNL': EmNL,
                    'Alpha': Alpha,
                    'fitness': solution_fitness,
                }, f, indent=4)

            self.ga.save('ga_instance')
            
            plt_fitness = self.ga.plot_fitness()
            plt_fitness.savefig('fitness.png')
    
            merged = self.compute_and_merge(solution)
    
            cols = ['Stress', 'StrainNonLinEl', 'StrainViscEl', 'TotStrain']
            fig, axes = plt.subplots(len(cols), 1, figsize=(10, 2.5 * len(cols)), sharex=True)
            
            for i, col in enumerate(cols):
                ax = axes[i]
                ax.plot(merged['time'].to_numpy(), merged[f'{col}_target'].to_numpy(), label='Target', color='black', linestyle='--')
                ax.plot(merged['time'].to_numpy(), merged[f'{col}_model'].to_numpy(), label='Model', color='blue')
                ax.set_ylabel(col)
                ax.legend()
                ax.grid(True)
            
            axes[-1].set_xlabel('Time')
            plt.tight_layout()
            plt.savefig('plots.png')

In [ ]:
def run_all_gas():
    for label, Rsig in parse_signal_file().items():
        ga_subj = GASubject(label, Rsig)
        ga_subj.run_ga()
        ga_subj.save_result()

run_all_gas()

## Generating testing data

In [12]:
def noise_for(val):
    noise_range = 0.005
    return random.triangular(val - noise_range, val + noise_range, val)

def generate_testing_data(Rsig):
    # === Parameters ===
    Np = 21  # Number of periods Tp
    Nsamp = 1000  # Number of samples per period
    
    # === Default Model Parameters ===
    Em = 1.0         # Modulus
    Tau_e = 50.0     # Relaxation time for strain
    Tau_s = 25.0     # Relaxation time for stress
    EmNL = 0.2       # Nonlinear modulus
    Alpha = 3.0      # Degree of nonlinearity
    Sig0 = 1.0       # Initial maximum stress
    Tp = 100.0       # Period
    
    # === Derived Values ===
    R = Sig0 / Tp
    dt = Tp / Nsamp
    t = 0.0
    FvalSig = 0.0
    FvalEps = 0.0
    t0 = 0.0
    
    # === Model functions ===
    def fSig(x, x1, Rs):
        return Rs * (x - x1)
    
    def fEps(x, x1, x2, Rs):
        return (Rs / Em) * ((x2 - x1) + (Tau_s - Tau_e) *
                            (np.exp(x2 / Tau_e) - np.exp(x1 / Tau_e)) *
                            np.exp(-x / Tau_e))
    
    def fEps0(x, Rs):
        return fEps(x, t0, x, Rs)
    
    def fEpsNL(Sig):
        return Sig / (EmNL * (1 + Alpha * Sig))
    
    def fEpsVE(x, i):
        eps = fEps0(x - (i - 1) * Tp, Rsig[i - 1] * R)
        for k in range(1, i):
            eps += fEps(x, (k - 1) * Tp, k * Tp, Rsig[k - 1] * R)
        return eps

    rows = []
    
    for i in range(1, Np + 1):
        for j in range(Nsamp):
            t += dt
            valSig = FvalSig + fSig(t, (i - 1) * Tp, Rsig[i - 1] * R)
            valEpsNL = fEpsNL(valSig)
            valEpsVE = fEpsVE(t, i)
            rows.append([
                t,
                valSig,
                valEpsNL,
                valEpsVE,
                noise_for(valEpsNL + valEpsVE),
            ])
        FvalSig = valSig

    df = pd.DataFrame(rows, columns=['time', 'Stress', 'StrainNonLinEl', 'StrainViscEl', 'TotStrain'])
    return df

def generate_all_testing_data():
    for label, Rsig in parse_signal_file().items():
        df = generate_testing_data(Rsig)
        with change_dir(label):
            df.to_csv('ZenerModel.dat', sep='\t', index=False)

# generate_all_testing_data()